<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 01. AlexNet

## ImageNet Classification with Deep Convolutional Neural Networks

- **원 논문:** [ImageNet Classification with Deep Convolutional Neural Networks](https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf)
- **저자 / 발표:** Alex Krizhevsky, Ilya Sutskever, Geoffrey E. Hinton · NeurIPS 2012 (2012)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** ReLU, channel-local LRN, overlapping pooling, 5-conv 경로를 재현한다.

**축소하거나 생략한 부분:** ImageNet 1.2M장과 약 60M parameter, 2-GPU 분할 대신 16×16 합성 도형과 작은 채널을 사용한다. 논문의 다섯 convolution 구조와 LRN 수식은 유지한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1: f(x)=max(0,x) | Task 1 | ReLU gradient assert |
| §3.3 Eq. (1): LRN | Task 1 paper_lrn | 수치 assert |
| §3.4: overlapping pooling z=3,s=2 | Task 2 | stride shape |
| §3.5 Fig. 2: five conv layers | Task 2–3 | layer/loss 검증 |

## 핵심 아이디어와 수식

$$b^i_{x,y}=a^i_{x,y}\left(k+\alpha\sum_j(a^j_{x,y})^2\right)^{-\beta}$$

**기호 해설:** a는 ReLU 활성값, j는 이웃 channel, k·α·β는 LRN 상수, b는 정규화 출력이다.

**코드 대응:** Task 1의 local_response_norm이 수식을 직접 계산하고 Task 2가 5-conv를 조립한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → 두 번의 overlap pooling → [B,12,4,4] → logits [B,4].

**포트폴리오 구현 범위:** 224×224 입력, data augmentation, multi-GPU는 제외하고 구조적 선택만 검증한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1: f(x)=max(0,x)
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** ReLU gradient assert

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class AlexNetConfig:
    channels: tuple = (8, 12, 16, 16, 12)
    num_classes: int = 4
    lrn_size: int = 5
    learning_rate: float = 0.01
    steps: int = 12


def local_response_norm(inputs, size, alpha, beta, k):
    half = size // 2
    squared = inputs.square().movedim(1, -1)
    squared = F.pad(squared, (half, half))
    local_energy = squared.unfold(-1, size, 1).sum(dim=-1)
    scale = (k + alpha * local_energy).pow(beta)
    return (inputs.movedim(1, -1) / scale).movedim(-1, 1)


config = AlexNetConfig()
spike = torch.zeros(1, 7, 1, 1, device=DEVICE)
spike[:, 3] = 2.0
normalized = local_response_norm(
    spike,
    config.lrn_size,
    1e-4,
    0.75,
    2.0,
)
expected = 2.0 / (2.0 + 4e-4) ** 0.75
assert torch.allclose(normalized[0, 3, 0, 0], spike.new_tensor(expected))

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.3 Eq. (1): LRN
- **노트북에서 구현할 부분:** Task 1 paper_lrn
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 수치 assert

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
class AlexNetMini(nn.Module):
    def __init__(self, config):
        super().__init__()
        c1, c2, c3, c4, c5 = config.channels
        self.conv1 = nn.Conv2d(1, c1, 3, padding=1)
        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1)
        self.conv3 = nn.Conv2d(c2, c3, 3, padding=1)
        self.conv4 = nn.Conv2d(c3, c4, 3, padding=1)
        self.conv5 = nn.Conv2d(c4, c5, 3, padding=1)
        self.classifier = nn.Linear(c5, config.num_classes)
        self.config = config

    def activate(self, inputs):
        return F.relu(inputs)

    def forward_features(self, inputs):
        features = self.activate(self.conv1(inputs))
        features = local_response_norm(
            features,
            self.config.lrn_size,
            1e-4,
            0.75,
            2.0,
        )
        features = F.max_pool2d(features, 3, stride=2, padding=1)
        features = self.activate(self.conv2(features))
        features = F.max_pool2d(features, 3, stride=2, padding=1)
        features = self.activate(self.conv3(features))
        features = self.activate(self.conv4(features))
        return self.activate(self.conv5(features))

    def forward(self, inputs):
        features = self.forward_features(inputs)
        return self.classifier(features.mean(dim=(2, 3)))


model = AlexNetMini(config).to(DEVICE)
assert model.config.lrn_size == config.lrn_size == 5
assert sum(isinstance(layer, nn.Conv2d) for layer in model.modules()) == 5
assert model(images[:2]).shape == (2, 4)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.5 Fig. 2: five conv layers
- **노트북에서 구현할 부분:** Task 2–3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** layer/loss 검증

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def classification_loss(logits, targets):
    return F.cross_entropy(logits, targets)


def train_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = classification_loss(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_classifier(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(inputs).argmax(dim=1)
    return float((predictions == targets).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    step_loss = train_step(model, optimizer, x_train, y_train)
    losses.append(step_loss)

accuracy = evaluate_classifier(
    model,
    images[test_idx],
    labels[test_idx],
)
assert min(losses[3:]) < losses[0]
assert 0.0 <= accuracy <= 1.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.title("AlexNet mini: accuracy={:.1%}".format(accuracy))
plt.tight_layout()
plt.show()
print(f"loss={losses[0]:.3f} -> {losses[-1]:.3f}")

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> ImageNet 1.2M장과 약 60M parameter, 2-GPU 분할 대신 16×16 합성 도형과 작은 채널을 사용한다. 논문의 다섯 convolution 구조와 LRN 수식은 유지한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.